# ASL v1 — STACKED: ROI-crop pretraining + ROI fine-tune (Colab T4)

Stacks our two winning levers: **motion-ROI crop** (kills background overfitting) +
**500-class encoder pretraining** (kills data scarcity). Same pipeline as the
center-crop run, but every cache is the ROI-cropped version.

### One-time setup
1. `Runtime → Change runtime type → T4 GPU`.
2. Upload **three files** to `MyDrive/asl-model/` (same folder as before):
   - `code_bundle.zip`            (refreshed — now includes norm_roi.json + finetune_roi.yaml)
   - `pretrain_roi_jpeg.npz`      (~1 GB — 500-class set, ROI-cropped, JPEG-packed)
   - `clips_roi.npz`             (~1.1 GB — 75-class set, ROI-cropped)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, zipfile, shutil
DRIVE = '/content/drive/MyDrive/asl-model'
os.makedirs('/content/work/artifacts/cache', exist_ok=True)
with zipfile.ZipFile(f'{DRIVE}/code_bundle.zip') as z:
    z.extractall('/content/work')
for f in ['pretrain_roi_jpeg.npz', 'clips_roi.npz']:
    shutil.copy(f'{DRIVE}/{f}', f'/content/work/artifacts/cache/{f}')
%cd /content/work
!pip -q install pyyaml onnx onnxruntime
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# Decode the ROI JPEG cache back to frames.dat
!PYTHONPATH=src python -m asl.pack_pretrain_jpeg --unpack \
    --in artifacts/cache/pretrain_roi_jpeg.npz --cache artifacts/cache/pretrain_roi

In [ ]:
# Pretrain encoder from scratch on the ROI-cropped 500-class set
!PYTHONPATH=src python -u -m asl.pretrain --cache artifacts/cache/pretrain_roi \
    --epochs 45 --warmup 4 --batch-size 64 --lr 0.004 \
    --out artifacts/checkpoints/pretrain
import shutil, os
os.makedirs('/content/drive/MyDrive/asl-model/out_roi', exist_ok=True)
shutil.copy('artifacts/checkpoints/pretrain/encoder.pt',
            '/content/drive/MyDrive/asl-model/out_roi/encoder.pt')
print('saved ROI encoder.pt to Drive/out_roi')

In [ ]:
# Fine-tune the 75-class head on ROI clips + ROI encoder
!PYTHONPATH=src python -u -m asl.train --config configs/finetune_roi.yaml
import shutil, os
OUT = '/content/drive/MyDrive/asl-model/out_roi'
for f in ['artifacts/checkpoints/finetune_roi/best.pt',
          'artifacts/checkpoints/finetune_roi/history.json']:
    shutil.copy(f, f'{OUT}/{os.path.basename(f)}')
print('stacked fine-tune done; results copied to Drive/out_roi')